In [ ]:
%env CUDA_VISIBLE_DEVICES=1
import torch
from torch import Tensor
import yaml
from models.vaectorgen import VAEctorGen
from models.vector_vae_nlayers import VectorVAEnLayers
from dataset import MNISTDataset
from PIL import Image
import matplotlib.pyplot as plt

# Vector_VQVAE

## Load model and checkpoint

In [ ]:
import yaml
from models import Vector_VQVAE
config_path = "/home/mfeuerpfeil/master/thesis/configs/SVG_VQVAE_Stage1.yaml"
with open(config_path, 'r') as file:
    try:
        config = yaml.safe_load(file)
    except yaml.YAMLError as exc:
        print(exc)

# adjust for inference
config['data_params']["max_shapes_per_svg"] = 200

device = "cuda" if torch.cuda.is_available() else "cpu"
model = Vector_VQVAE(**config['model_params']).to(device)

ckpt_path = "/scratch2/moritz_logs/SVG_VQVAE/Stage1/glyphazzn_full_single_code/checkpoints/last.ckpt"
state_dict = torch.load(ckpt_path, map_location=device)["state_dict"]
model.load_state_dict({k.replace("model.", ""): v for k, v in state_dict.items()})

In [ ]:
encoder_params = sum(p.numel() for p in model.encoder.parameters()) / 1_000_000
decoder_params = sum(p.numel() for p in model.decoder.parameters()) / 1_000_000

print(f"Number of parameters in the encoder: {encoder_params:.2f} million")
print(f"Number of parameters in the decoder: {decoder_params:.2f} million")

total_params = sum(p.numel() for p in model.parameters()) / 1_000_000
print(f"Total number of parameters in the model: {total_params:.2f} million")

## prepare data

In [ ]:
from dataset import GlyphazznStage1Datamodule
dm = GlyphazznStage1Datamodule(**config['data_params'])
dm.setup()
val_dl = dm.val_dataloader()
dataset = dm.val_dataset

## inference

In [ ]:
from tqdm import tqdm
all_losses = []
model = model.eval()
with torch.no_grad():
    for batch in tqdm(val_dl, total=len(val_dl)):
        all_center_shapes, _, positions = batch
        all_center_shapes = all_center_shapes.to(device)
        positions = positions.to(device)
        out, logging_dict = model.forward(all_center_shapes)
        reconstructions=out[0][:,:3,:,:]
        loss_dict = model.loss_function(
                reconstructions=reconstructions,
                gt_images=all_center_shapes,
                vq_loss=torch.tensor(0.0).to(device),
        )
        # reconstructed_drawing, rasterized_reconstructions = model.reconstruct(all_center_shapes, positions, dm.individual_max_length +2, dm.stroke_width)
        all_losses.append(loss_dict["loss"])

In [ ]:
sum(all_losses) / len(all_losses)

## visualization

In [ ]:
import torchvision.utils as vutils
def visualize_comparison(reconstructions, gt, save_path=None):
    # Create grids of images
    reconstructions_grid = vutils.make_grid(reconstructions, nrow=16, padding=2, normalize=True)
    gt_grid = vutils.make_grid(gt, nrow=16, padding=2, normalize=True)

    # Create a side-by-side comparison
    comparison = torch.cat((gt_grid, reconstructions_grid), dim=2)

    # Convert the comparison to a numpy array
    comparison_np = comparison.permute(1, 2, 0).cpu().numpy()

    # Display the comparison
    plt.imshow(comparison_np)
    plt.axis('off')
    if save_path is not None:
        plt.savefig(save_path, dpi=600)
    plt.show()

visualize_comparison(reconstructions, all_center_shapes, save_path="images/val_recons_vqvae_stage1.png")

In [ ]:
import random
random_idx = random.randint(0, len(dm.val_dataset))
dataset = dm.val_dataset
w = 480

patches, labels, positions = dataset[random_idx]
patches = patches.to(device)
positions = positions.to(device)

gt = dataset._get_full_svg_drawing(random_idx, width=w, as_tensor=True)
recons_drawing, recons_rastered_drawing = model.reconstruct(patches, positions, dataset.individual_max_length +2, dataset.stroke_width, rendered_w=w)

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Plot reconstructed drawing
axes[1].imshow(recons_rastered_drawing.permute(1, 2, 0))
axes[1].set_title('Reconstructed SVG')

# Plot ground truth drawing
axes[0].imshow(gt.permute(1, 2, 0))
axes[0].set_title('Ground Truth SVG')

# Remove axis labels
for ax in axes:
    ax.axis('off')

# Adjust spacing between subplots
plt.tight_layout()

# Show the plot
plt.show()

# Stage 2

In [ ]:
%env CUDA_VISIBLE_DEVICES=1
import os
import yaml
from dataset import VQDataModule
from models import VQ_SVG_Stage2, Vector_VQVAE
from tokenizer import VQTokenizer
from experiment import SVG_VQVAE_Stage2_Experiment
import torch
import random
import matplotlib.pyplot as plt
import numpy as np
import torchvision.utils as vutils
from PIL import Image
from torch import Tensor


torch.cuda.is_available()
from utils import calculate_global_positions, shapes_to_drawing, drawing_to_tensor
from svg_fixing import get_fixed_svg_drawing, get_fixed_svg_render

## Load everything

In [ ]:
CONFIG_PATH = "configs/SVG_VQVAE_Stage2.yaml"
CKPT_PATH = "/scratch2/moritz_logs/SVG_VQVAE/Stage2/glyphazzn_full_single_code/checkpoints/last-v3.ckpt"

device = "cuda" if torch.cuda.is_available() else "cpu"

with open(CONFIG_PATH, 'r') as file:
    try:
        config = yaml.safe_load(file)
    except yaml.YAMLError as exc:
        print(exc)

vq_model = Vector_VQVAE(**config['stage1_params'], device = device)
state_dict = torch.load(config['stage1_params']["checkpoint_path"])["state_dict"]
try:
    vq_model.load_state_dict(state_dict)
except:
    vq_model.load_state_dict({k.replace("model.", ""): v for k, v in state_dict.items()})
vq_model = vq_model.eval()
tokenizer = VQTokenizer(vq_model, config["data_params"]["width"], 1, "bert-base-uncased", device = device)
model = VQ_SVG_Stage2(tokenizer, **config['model_params'], device = device)
state_dict = torch.load(CKPT_PATH)["state_dict"]
try:
    model.load_state_dict(state_dict)
except:
    new_dict = {k.replace("model.", ""): v for k, v in state_dict.items()}
    new_dict = {k.replace("transformer.", "transformer.model."): v for k, v in new_dict.items()}
    model.load_state_dict(new_dict)

model = model.eval().to(device)
data = VQDataModule(**config["data_params"], context_length=config['model_params']['max_seq_len'])
data.setup()

## generate

### prompts from dataloader

In [ ]:
generations = []
captions = []
ds = data.train_dataset

with torch.no_grad():
    for i in range(25):
        rand_idx = random.randint(0, len(ds))
        text_tokens, attention_mask, vq_tokens, vq_targets, pad_token = ds[rand_idx]

        generation, reason = model.generate(text_tokens.unsqueeze(0).to(device),
                                    attention_mask.unsqueeze(0).to(device), 
                                    torch.ones((1, 1), device = device, dtype=torch.int64) * tokenizer.special_token_mapping.get("<BOS>"))
        generations.append(generation)
        captions.append(tokenizer.decode_text(text_tokens))

In [ ]:
fig, axes = plt.subplots(5, 5, figsize=(20, 20))
for i, ax in enumerate(axes.flatten()):
    bezier_points, positions = tokenizer.decode(generations[i][0], ignore_special_tokens=True)
    ax.imshow(get_fixed_svg_render(bezier_points, positions, method="min_dist_clip").permute(1, 2, 0))
    ax.set_title(captions[i])
    ax.axis('off')

fig.suptitle(f"Generated (and fixed) SVGs - {CKPT_PATH.split('/')[-1].split('.')[0]}")
# plt.savefig(f"images/stage2_images/{CKPT_PATH.split('/')[-1].split('.')[0]}_{rand_idx}.png", dpi=600)
plt.show()

### custom prompts

In [ ]:
custom_prompts = ["a in normal font", "capital B in normal font", "capital X in normal font", "z in normal font", ""]
generations = []
captions = []
with torch.no_grad():
    for i, prompt in enumerate(custom_prompts):
        text_tokens = tokenizer.tokenize_text(prompt)
        attention_mask = torch.ones(len(text_tokens), device = device, dtype=torch.int64)

        generation, reason = model.generate(text_tokens.unsqueeze(0).to(device),
                                    attention_mask.unsqueeze(0).to(device), 
                                    torch.ones((1, 1), device = device, dtype=torch.int64) * tokenizer.special_token_mapping.get("<BOS>"))
        generations.append(generation)
        captions.append(tokenizer.decode_text(text_tokens))

fig, axes = plt.subplots(1, len(custom_prompts), figsize=(5 * len(custom_prompts), 10))
for i, ax in enumerate(axes.flatten()):
    bezier_points, positions = tokenizer.decode(generations[i][0], ignore_special_tokens=True)
    ax.imshow(get_fixed_svg_render(bezier_points, positions, method="min_dist_clip").permute(1, 2, 0))
    ax.set_title(captions[i])
    ax.axis('off')

fig.suptitle(f"Generated (and fixed) SVGs - {CKPT_PATH.split('/')[-1].split('.')[0]}")
# plt.savefig(f"images/stage2_images/{CKPT_PATH.split('/')[-1].split('.')[0]}_{rand_idx}.png", dpi=600)
plt.show()

## visualize different fixing algorithms

In [ ]:
rand_idx = random.randint(0, len(data.val_dataset))
text_tokens, attention_mask, vq_tokens, vq_targets, pad_token = data.val_dataset[rand_idx]

generation, reason = model.generate(text_tokens.unsqueeze(0).to(device),
                                    attention_mask.unsqueeze(0).to(device), 
                                    torch.ones((1, 1), device = device, dtype=torch.int64) * tokenizer.special_token_mapping.get("<BOS>"))

bezier_points, positions = tokenizer.decode(generation, ignore_special_tokens=True)
global_shapes = calculate_global_positions(bezier_points, 7.5 + 2, positions)[:,0]

clipped_img = get_fixed_svg_render(bezier_points, positions, "clip")
interpolated_img = get_fixed_svg_render(bezier_points, positions, "interpolate")
min_dist_clip_img = get_fixed_svg_render(bezier_points, positions, "min_dist_clip")
min_dist_interpolate_img = get_fixed_svg_render(bezier_points, positions, "min_dist_interpolate")

fig, axes = plt.subplots(1, 5, figsize=(15, 5))
unfixed_img = tokenizer._tokens_to_image_tensor(generation)
axes[0].imshow(unfixed_img.permute(1, 2, 0))
axes[0].set_title('Unprocessed generated SVG')
axes[1].imshow(clipped_img.permute(1, 2, 0))
axes[1].set_title('Naive clip SVG')
axes[2].imshow(interpolated_img.permute(1, 2, 0))
axes[2].set_title('Naive interpolate SVG')
axes[3].imshow(min_dist_clip_img.permute(1, 2, 0))
axes[3].set_title('Min dist clip SVG')
axes[4].imshow(min_dist_interpolate_img.permute(1, 2, 0))
axes[4].set_title('Min dist interpolate SVG')

for ax in axes:
    ax.axis('off')

fig.suptitle(f"Comparing post-processing for SVG generations\nPrompt: {tokenizer.decode_text(text_tokens)}")
plt.show()

# Im2Vec and VAEctorGen

In [ ]:
im2vec_config_path = "/home/mfeuerpfeil/master/thesis/configs/Im2Vec_mnist.yaml"
im2vec_model_path = "/scratch2/moritz_logs/Im2Vec_MNIST/checkpoints/last.ckpt"

vaectorgen_config_path = "/home/mfeuerpfeil/master/thesis/configs/VAEctorGen_mnist.yaml"
vaectorgen_model_path = "/scratch2/moritz_logs/VAEctorgen_mnist/checkpoints/last-v2.ckpt"

In [ ]:
with open(im2vec_config_path, "r") as f:
    try:
        im2vec_config = yaml.safe_load(f)
    except yaml.YAMLError as exc:
        print(exc)

with open(vaectorgen_config_path, "r") as f:
    try:
        vaectorgen_config = yaml.safe_load(f)
    except yaml.YAMLError as exc:
        print(exc)

In [ ]:
im2vec = VectorVAEnLayers(**im2vec_config["model_params"])
vaectorgen = VAEctorGen(**vaectorgen_config["model_params"])

In [ ]:
def sample_to_image(sample:Tensor):
    if(sample.requires_grad):
        sample = sample.detach().cpu()
    
    # collapse batch-first dimension
    if(sample.shape[0] == 1):
        sample = sample.squeeze(0)

    # permute if new 0-th dimension is channels
    if(sample.shape[0] == 3 or sample.shape[0] == 4):
        return sample.permute(1,2,0)

    return sample

In [ ]:
def show_images(image_tensor1, image_tensor2):
    fig, axs = plt.subplots(1, 2)
    axs[0].imshow(sample_to_image(image_tensor1))
    axs[1].imshow(sample_to_image(image_tensor2))
    plt.show()

# Reconstruction

In [ ]:
ds = MNISTDataset(**im2vec_config["data_params"])
ds.setup()
dataloader = ds.test_dataloader()
imgs, labels = next(iter(dataloader))
with torch.no_grad():
    reconstructions = im2vec.generate(imgs)

In [ ]:
for i in range(5):
    show_images(imgs[i], reconstructions[i])

# Sampling

In [ ]:
plt.imshow(sample_to_image(vaectorgen.sample(1, "cpu")))